In [0]:
%sql

-- ATLAS Marine — Notebook 3: Gold Layer
-- Star Schema dimensional model
-- Five tables: four dimensions + one central fact table

-- SECTION 1 — DIM_SPECIES
-- Four rows — one per species — static reference table
-- Never changes unless a new species is added to ATLAS


DROP TABLE IF EXISTS mcphersonsharyn_gold.dim_species;

CREATE TABLE mcphersonsharyn_gold.dim_species
(
  species_id        INT,
  species_common    STRING,
  species_latin     STRING,
  iucn_status       STRING,
  feeding_type      STRING,
  study_region      STRING,
  ocean_basin       STRING,
  individual_count  INT,
  total_observations INT,
  tracking_start    INT,
  tracking_end      INT
)
USING DELTA;
-- Static reference table — 4 rows one per ATLAS species
-- species_id is the surrogate key used in fact table
-- All attributes confirmed from Movebank metadata

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.dim_species VALUES
(1, 'Reef Manta Ray',  'Mobula alfredi',          'Vulnerable',        'Filter feeder',  'Raja Ampat Indonesia',      'Indo-Pacific',    25, 3224,  2000, 2020),
(2, 'Whale Shark',     'Rhincodon typus',           'Endangered',        'Filter feeder',  'Gulf of Mexico',            'Atlantic',        41, 3382,  2003, 2019),
(3, 'Sperm Whale',     'Physeter macrocephalus',    'Vulnerable',        'Deep diver',     'Gulf of Mexico',            'Atlantic',        17, 7759,  2001, 2015),
(4, 'Green Turtle',    'Chelonia mydas',            'Endangered',        'Herbivore',      'Indian Ocean Chagos',       'Indian Ocean',    33, 55411, 1998, 2020);
-- Four ATLAS species with key attributes
-- species_id 1-4 used as foreign key in fact table



In [0]:
%sql
SELECT * FROM mcphersonsharyn_gold.dim_species
ORDER BY species_id;
-- Shows all four species rows
-- Confirms species_id, names, IUCN status and observation counts

In [0]:
%sql

-- SECTION 2 — DIM_DATE
-- One row per year-month combination
-- Enriched with ENSO climate context from enso_silver



DROP TABLE IF EXISTS mcphersonsharyn_gold.dim_date;

CREATE TABLE mcphersonsharyn_gold.dim_date
USING DELTA AS
SELECT
  CONCAT(CAST(year AS STRING), '-', LPAD(CAST(month AS STRING), 2, '0')) AS date_id,
  year,
  month,
  CASE month
    WHEN 1 THEN 'January'   WHEN 2 THEN 'February'  WHEN 3 THEN 'March'
    WHEN 4 THEN 'April'     WHEN 5 THEN 'May'        WHEN 6 THEN 'June'
    WHEN 7 THEN 'July'      WHEN 8 THEN 'August'     WHEN 9 THEN 'September'
    WHEN 10 THEN 'October'  WHEN 11 THEN 'November'  WHEN 12 THEN 'December'
  END AS month_name,
  CASE
    WHEN month IN (1, 2, 3) THEN 1
    WHEN month IN (4, 5, 6) THEN 2
    WHEN month IN (7, 8, 9) THEN 3
    ELSE 4
  END AS quarter,
  CASE
    WHEN month IN (12, 1, 2) THEN 'Boreal Winter'
    WHEN month IN (3, 4, 5)  THEN 'Boreal Spring'
    WHEN month IN (6, 7, 8)  THEN 'Boreal Summer'
    ELSE 'Boreal Autumn'
  END AS season,
  ROUND(mei_value, 3) AS mei_value,
  enso_phase,
  CASE WHEN enso_phase = 'El Nino' THEN TRUE ELSE FALSE END AS is_el_nino,
  CASE WHEN enso_phase = 'La Nina' THEN TRUE ELSE FALSE END AS is_la_nina
FROM mcphersonsharyn_silver.enso_silver
ORDER BY year, month;
-- date_id format yyyy-MM used as foreign key in fact table
-- LPAD ensures month is always 2 digits (01 not 1)
-- season uses Boreal (Northern Hemisphere) convention
-- is_el_nino and is_la_nina boolean flags for easy

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  min(date_id) AS earliest,
  max(date_id) AS latest,
  count(CASE WHEN enso_phase = 'El Nino' THEN 1 END) AS el_nino_months,
  count(CASE WHEN enso_phase = 'La Nina' THEN 1 END) AS la_nina_months,
  count(CASE WHEN enso_phase = 'Neutral' THEN 1 END) AS neutral_months
FROM mcphersonsharyn_gold.dim_date;
-- Expected: 576 rows, 1979-01 to 2026-12
-- Three ENSO phases all present

In [0]:
%sql

-- SECTION 3 — DIM_LOCATION
-- One row per H3 resolution 4 cell
-- Derived from SST Silver — all unique H3 cells in study region



DROP TABLE IF EXISTS mcphersonsharyn_gold.dim_location;

CREATE TABLE mcphersonsharyn_gold.dim_location
USING DELTA AS
SELECT DISTINCT
  h3_res4,
  h3_res5,
  h3_longlatash3(longitude, latitude, 3) AS h3_res3,
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CASE
    WHEN longitude BETWEEN -98 AND -73 AND latitude BETWEEN 18 AND 22
      THEN 'Gulf of Mexico'
    WHEN longitude BETWEEN 125 AND 135 AND latitude BETWEEN -5 AND 2
      THEN 'Raja Ampat Indonesia'
    WHEN longitude BETWEEN 70 AND 75 AND latitude BETWEEN -8 AND -5
      THEN 'Chagos Archipelago'
    WHEN longitude BETWEEN -180 AND -73
      THEN 'Western Atlantic and Caribbean'
    WHEN longitude BETWEEN 126 AND 180
      THEN 'Indo-Pacific'
    ELSE 'Open Ocean'
  END AS study_region,
  CASE
    WHEN longitude BETWEEN -180 AND -73
      THEN 'Atlantic'
    WHEN longitude BETWEEN 20 AND 125
      THEN 'Indian Ocean'
    WHEN longitude BETWEEN 125 AND 180
      THEN 'Indo-Pacific'
    ELSE 'Pacific'
  END AS ocean_basin
FROM mcphersonsharyn_silver.sst_silver
ORDER BY latitude, longitude;
-- One row per unique H3 cell in the SST Silver dataset
-- Geographic context assigned by coordinate ranges
-- study_region identifies the four ATLAS species habitats
-- ocean_basin provides broader geographic classification
-- h3_res3 added for CMEMS nearest neighbour joins

In [0]:
%sql
SELECT
  ocean_basin,
  study_region,
  count(*) AS h3_cells
FROM mcphersonsharyn_gold.dim_location
GROUP BY ocean_basin, study_region
ORDER BY ocean_basin, study_region;
-- Shows geographic distribution of H3 cells
-- Confirms all four species study regions are represented

In [0]:
%sql

-- SECTION 4 — DIM_ENVIRONMENT
-- One row per H3 cell per year per month
-- All environmental variables in one place
-- Composite key: h3_res4 + year + month


DROP TABLE IF EXISTS mcphersonsharyn_gold.dim_environment;

CREATE TABLE mcphersonsharyn_gold.dim_environment
USING DELTA AS
SELECT
  h3_res4,
  year,
  month,
  sea_surface_temperature,
  chlorophyll_concentration,
  current_speed,
  sea_level_anomaly,
  mixed_layer_depth,
  salinity,
  ocean_temperature,
  microplastic_concentration,
  microplastics_source,
  degree_heating_week,
  bleaching_alert_area,
  bleaching_alert_category,
  habitat_stress_index
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Direct copy from environmental_conditions_silver
-- All environmental variables consolidated in one dimension
-- Joins to fact table on h3_res4 + year + month
-- 2,795,120 rows — one per H3 cell per year per month

In [0]:
%sql
SELECT
  'dim_environment' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT h3_res4) AS distinct_h3_cells,
  count(DISTINCT year) AS distinct_years,
  count(CASE WHEN sea_surface_temperature IS NOT NULL THEN 1 END) AS rows_with_sst,
  count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) AS rows_with_cc,
  count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) AS rows_with_bleaching,
  count(DISTINCT habitat_stress_index) AS stress_index_values
FROM mcphersonsharyn_gold.dim_environment;
-- Confirms dim_environment built correctly
-- distinct_h3_cells should be 6,946
-- distinct_years should be 34

In [0]:
%sql
SELECT *
FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
LIMIT 3;
-- Shows column names in manta ray Bronze table
-- Need exact column names before building fact table
-- Look for individual ID, timestamp, latitude, longitude columns

In [0]:
%sql
-- ═══════════════════════════════════════════════════════════════
-- SECTION 5 — FACT_SPECIES_OBSERVATIONS
-- Central fact table — one row per species tracking observation
-- Joins all four species Bronze tables with UNION ALL
-- Enriched with H3 index and date_id for dimension joins
-- ═══════════════════════════════════════════════════════════════

DROP TABLE IF EXISTS mcphersonsharyn_gold.fact_species_observations;

CREATE TABLE mcphersonsharyn_gold.fact_species_observations
USING DELTA AS
WITH all_species AS (
  -- Reef Manta Ray
  SELECT
    CAST(`event-id` AS BIGINT) AS observation_id,
    1 AS species_id,
    CAST(`individual-local-identifier` AS STRING) AS animal_id,
    CAST(`location-lat` AS DOUBLE) AS latitude,
    CAST(`location-long` AS DOUBLE) AS longitude,
    TO_TIMESTAMP(`timestamp`) AS observation_timestamp,
    YEAR(TO_TIMESTAMP(`timestamp`)) AS year,
    MONTH(TO_TIMESTAMP(`timestamp`)) AS month,
    CONCAT(
      CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
      LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
    ) AS date_id,
    locality,
    CAST(`manually-marked-outlier` AS BOOLEAN) AS algorithm_marked_outlier,
    `sensor-type` AS sensor_type
  FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
  WHERE `location-lat` IS NOT NULL
  AND `location-long` IS NOT NULL

  UNION ALL

  -- Whale Shark
  SELECT
    CAST(`event-id` AS BIGINT),
    2,
    CAST(`individual-local-identifier` AS STRING),
    CAST(`location-lat` AS DOUBLE),
    CAST(`location-long` AS DOUBLE),
    TO_TIMESTAMP(`timestamp`),
    YEAR(TO_TIMESTAMP(`timestamp`)),
    MONTH(TO_TIMESTAMP(`timestamp`)),
    CONCAT(
      CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
      LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
    ),
    locality,
    CAST(`manually-marked-outlier` AS BOOLEAN),
    `sensor-type`
  FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
  WHERE `location-lat` IS NOT NULL
  AND `location-long` IS NOT NULL

  UNION ALL

  -- Sperm Whale
  SELECT
    CAST(`event-id` AS BIGINT),
    3,
    CAST(`individual-local-identifier` AS STRING),
    CAST(`location-lat` AS DOUBLE),
    CAST(`location-long` AS DOUBLE),
    TO_TIMESTAMP(`timestamp`),
    YEAR(TO_TIMESTAMP(`timestamp`)),
    MONTH(TO_TIMESTAMP(`timestamp`)),
    CONCAT(
      CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
      LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
    ),
    locality,
    CAST(`manually-marked-outlier` AS BOOLEAN),
    `sensor-type`
  FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
  WHERE `location-lat` IS NOT NULL
  AND `location-long` IS NOT NULL

  UNION ALL

  -- Green Turtle
  SELECT
    CAST(`event-id` AS BIGINT),
    4,
    CAST(`individual-local-identifier` AS STRING),
    CAST(`location-lat` AS DOUBLE),
    CAST(`location-long` AS DOUBLE),
    TO_TIMESTAMP(`timestamp`),
    YEAR(TO_TIMESTAMP(`timestamp`)),
    MONTH(TO_TIMESTAMP(`timestamp`)),
    CONCAT(
      CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
      LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
    ),
    locality,
    CAST(`manually-marked-outlier` AS BOOLEAN),
    `sensor-type`
  FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
  WHERE `location-lat` IS NOT NULL
  AND `location-long` IS NOT NULL
)
SELECT
  observation_id,
  species_id,
  animal_id,
  date_id,
  year,
  month,
  observation_timestamp,
  latitude,
  longitude,
  h3_longlatash3(longitude, latitude, 4) AS h3_res4,
  h3_longlatash3(longitude, latitude, 5) AS h3_res5,
  locality,
  algorithm_marked_outlier,
  sensor_type
FROM all_species;
-- UNION ALL combines all four species into one fact table
-- species_id 1-4 links to dim_species
-- date_id links to dim_date
-- h3_res4 links to dim_location and dim_environment
-- H3 calculated at point of observation for precise spatial join

In [0]:
%sql
DESCRIBE TABLE mcphersonsharyn_bronze.bronze_whale_shark_raw;
-- Shows exact column names for whale shark
-- Looking for the locality equivalent column name

In [0]:
%sql
DESCRIBE TABLE mcphersonsharyn_bronze.bronze_sperm_whale_raw;

In [0]:
%sql
DESCRIBE TABLE mcphersonsharyn_bronze.bronze_green_turtle_raw;
-- Checking green turtle column names
-- Need to confirm locality and outlier column names

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_gold.fact_species_observations;

CREATE TABLE mcphersonsharyn_gold.fact_species_observations
USING DELTA AS
WITH all_species AS (

  -- Reef Manta Ray
  SELECT
    CAST(`event-id` AS BIGINT) AS observation_id,
    1 AS species_id,
    CAST(`individual-local-identifier` AS STRING) AS animal_id,
    CAST(`location-lat` AS DOUBLE) AS latitude,
    CAST(`location-long` AS DOUBLE) AS longitude,
    TO_TIMESTAMP(`timestamp`) AS observation_timestamp,
    YEAR(TO_TIMESTAMP(`timestamp`)) AS year,
    MONTH(TO_TIMESTAMP(`timestamp`)) AS month,
    CONCAT(
      CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
      LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
    ) AS date_id,
    locality,
    CAST(`manually-marked-outlier` AS BOOLEAN) AS algorithm_marked_outlier,
    `sensor-type` AS sensor_type,
    NULL AS argos_lc
  FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
  WHERE `location-lat` IS NOT NULL
  AND `location-long` IS NOT NULL

  UNION ALL

  -- Whale Shark
  SELECT
    CAST(`event-id` AS BIGINT),
    2,
    CAST(`individual-local-identifier` AS STRING),
    CAST(`location-lat` AS DOUBLE),
    CAST(`location-long` AS DOUBLE),
    TO_TIMESTAMP(`timestamp`),
    YEAR(TO_TIMESTAMP(`timestamp`)),
    MONTH(TO_TIMESTAMP(`timestamp`)),
    CONCAT(
      CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
      LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
    ),
    NULL AS locality,
    CAST(`algorithm-marked-outlier` AS BOOLEAN),
    `sensor-type`,
    `argos:lc`
  FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
  WHERE `location-lat` IS NOT NULL
  AND `location-long` IS NOT NULL

  UNION ALL

  -- Sperm Whale
  SELECT
    CAST(`event-id` AS BIGINT),
    3,
    CAST(`individual-local-identifier` AS STRING),
    CAST(`location-lat` AS DOUBLE),
    CAST(`location-long` AS DOUBLE),
    TO_TIMESTAMP(`timestamp`),
    YEAR(TO_TIMESTAMP(`timestamp`)),
    MONTH(TO_TIMESTAMP(`timestamp`)),
    CONCAT(
      CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
      LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
    ),
    NULL AS locality,
    CAST(`manually-marked-outlier` AS BOOLEAN),
    `sensor-type`,
    NULL AS argos_lc
  FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
  WHERE `location-lat` IS NOT NULL
  AND `location-long` IS NOT NULL

  UNION ALL

  -- Green Turtle
  SELECT
    CAST(`event-id` AS BIGINT),
    4,
    CAST(`individual-local-identifier` AS STRING),
    CAST(`location-lat` AS DOUBLE),
    CAST(`location-long` AS DOUBLE),
    TO_TIMESTAMP(`timestamp`),
    YEAR(TO_TIMESTAMP(`timestamp`)),
    MONTH(TO_TIMESTAMP(`timestamp`)),
    CONCAT(
      CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
      LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
    ),
    NULL AS locality,
    CAST(`algorithm-marked-outlier` AS BOOLEAN),
    `sensor-type`,
    `argos:lc`
  FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
  WHERE `location-lat` IS NOT NULL
  AND `location-long` IS NOT NULL
)
SELECT
  observation_id,
  species_id,
  animal_id,
  date_id,
  year,
  month,
  observation_timestamp,
  latitude,
  longitude,
  h3_longlatash3(longitude, latitude, 4) AS h3_res4,
  h3_longlatash3(longitude, latitude, 5) AS h3_res5,
  locality,
  algorithm_marked_outlier,
  sensor_type,
  argos_lc
FROM all_species;
-- NULL used for columns that don't exist in some species tables
-- locality only available for manta ray
-- argos_lc only available for whale shark and green turtle
-- algorithm_marked_outlier handles both manually and algorithm flagged outliers
-- H3 calculated at exact GPS position for precise spatial joins

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_gold.fact_species_observations;

CREATE TABLE mcphersonsharyn_gold.fact_species_observations
(
  observation_id           BIGINT,
  species_id               INT,
  animal_id                STRING,
  date_id                  STRING,
  year                     INT,
  month                    INT,
  observation_timestamp    TIMESTAMP,
  latitude                 DOUBLE,
  longitude                DOUBLE,
  h3_res4                  STRING,
  h3_res5                  STRING,
  locality                 STRING,
  algorithm_marked_outlier BOOLEAN,
  sensor_type              STRING,
  argos_lc                 STRING
)
USING DELTA;
-- Empty fact table with correct schema
-- INSERT INTO will populate from all four species Bronze tables

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  CAST(`event-id` AS BIGINT) AS observation_id,
  1 AS species_id,
  CAST(`individual-local-identifier` AS STRING) AS animal_id,
  CONCAT(
    CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
  ) AS date_id,
  YEAR(TO_TIMESTAMP(`timestamp`)) AS year,
  MONTH(TO_TIMESTAMP(`timestamp`)) AS month,
  TO_TIMESTAMP(`timestamp`) AS observation_timestamp,
  CAST(`location-lat` AS DOUBLE) AS latitude,
  CAST(`location-long` AS DOUBLE) AS longitude,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
  locality,
  CAST(`manually-marked-outlier` AS BOOLEAN) AS algorithm_marked_outlier,
  `sensor-type` AS sensor_type,
  NULL AS argos_lc
FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL;
-- Inserts Reef Manta Ray observations -- species_id = 1
-- Expected: 3,224 rows

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  CAST(`event-id` AS BIGINT) AS observation_id,
  2 AS species_id,
  CAST(`individual-local-identifier` AS STRING) AS animal_id,
  CONCAT(
    CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
  ) AS date_id,
  YEAR(TO_TIMESTAMP(`timestamp`)) AS year,
  MONTH(TO_TIMESTAMP(`timestamp`)) AS month,
  TO_TIMESTAMP(`timestamp`) AS observation_timestamp,
  CAST(`location-lat` AS DOUBLE) AS latitude,
  CAST(`location-long` AS DOUBLE) AS longitude,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
  NULL AS locality,
  CAST(`algorithm-marked-outlier` AS BOOLEAN) AS algorithm_marked_outlier,
  `sensor-type` AS sensor_type,
  `argos:lc` AS argos_lc
FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL;
-- Inserts Whale Shark observations -- species_id = 2
-- Expected: 3,382 rows

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  CAST(`event-id` AS BIGINT) AS observation_id,
  3 AS species_id,
  CAST(`individual-local-identifier` AS STRING) AS animal_id,
  CONCAT(
    CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
  ) AS date_id,
  YEAR(TO_TIMESTAMP(`timestamp`)) AS year,
  MONTH(TO_TIMESTAMP(`timestamp`)) AS month,
  TO_TIMESTAMP(`timestamp`) AS observation_timestamp,
  CAST(`location-lat` AS DOUBLE) AS latitude,
  CAST(`location-long` AS DOUBLE) AS longitude,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
  NULL AS locality,
  CAST(`manually-marked-outlier` AS BOOLEAN) AS algorithm_marked_outlier,
  `sensor-type` AS sensor_type,
  NULL AS argos_lc
FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL;
-- Inserts Sperm Whale observations -- species_id = 3
-- Expected: 7,759 rows

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  CAST(`event-id` AS BIGINT) AS observation_id,
  4 AS species_id,
  CAST(`individual-local-identifier` AS STRING) AS animal_id,
  CONCAT(
    CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')
  ) AS date_id,
  YEAR(TO_TIMESTAMP(`timestamp`)) AS year,
  MONTH(TO_TIMESTAMP(`timestamp`)) AS month,
  TO_TIMESTAMP(`timestamp`) AS observation_timestamp,
  CAST(`location-lat` AS DOUBLE) AS latitude,
  CAST(`location-long` AS DOUBLE) AS longitude,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
  NULL AS locality,
  CAST(`algorithm-marked-outlier` AS BOOLEAN) AS algorithm_marked_outlier,
  `sensor-type` AS sensor_type,
  `argos:lc` AS argos_lc
FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL;
-- Inserts Green Turtle observations -- species_id = 4
-- Expected: 55,411 rows

In [0]:
%sql
SELECT
  s.species_common,
  count(*) AS total_observations,
  count(DISTINCT f.animal_id) AS unique_animals,
  min(f.year) AS earliest_year,
  max(f.year) AS latest_year,
  count(CASE WHEN f.algorithm_marked_outlier = TRUE THEN 1 END) AS outliers
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s
  ON f.species_id = s.species_id
GROUP BY s.species_common
ORDER BY s.species_common;
-- Validates all four species present with correct row counts
-- Confirms dim_species join works correctly
-- Shows outlier counts per species

In [0]:
%sql
SELECT
  d.enso_phase,
  s.species_common,
  count(DISTINCT f.animal_id) AS unique_animals,
  count(*) AS total_observations,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.chlorophyll_concentration), 4) AS avg_cc,
  round(avg(e.habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s
  ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d
  ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE d.enso_phase IS NOT NULL
AND s.species_common = 'Reef Manta Ray'
GROUP BY d.enso_phase, s.species_common
ORDER BY avg_sst DESC;
-- THE KEY ATLAS FINDING ON THE GOLD STAR SCHEMA
-- La Nina should show higher SST than El Nino in Raja Ampat
-- This confirms the finding using proper dimensional model joins
-- fact → dim_date for ENSO phase
-- fact → dim_environment for SST
-- fact → dim_species for species filter

In [0]:
%sql
SELECT
  count(*) AS total_fact_rows,
  count(DISTINCT f.h3_res4) AS fact_h3_cells,
  count(e.h3_res4) AS rows_with_environment,
  round(count(e.h3_res4) * 100.0 / count(*), 1) AS environment_join_pct
FROM mcphersonsharyn_gold.fact_species_observations f
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month;
-- Shows what percentage of fact rows join to dim_environment
-- If very low the species H3 cells don't overlap with SST H3 cells
-- This is the Indian Ocean gap affecting green turtle data

In [0]:
%sql
SELECT
  s.species_common,
  count(*) AS total_obs,
  count(e.h3_res4) AS obs_with_environment,
  round(count(e.h3_res4) * 100.0 / count(*), 1) AS join_pct
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
GROUP BY s.species_common
ORDER BY join_pct DESC;
-- Shows which species have environment data joining
-- Gulf of Mexico species should join well
-- Indian Ocean species will have zero joins

In [0]:
%sql
SELECT
  s.species_common,
  round(min(f.longitude), 1) AS min_lon,
  round(max(f.longitude), 1) AS max_lon,
  round(min(f.latitude), 1) AS min_lat,
  round(max(f.latitude), 1) AS max_lat
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
GROUP BY s.species_common
ORDER BY s.species_common;
-- Shows exact coordinate ranges for each species
-- Confirms whether species locations fall within SST coverage

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/sst-gulfofmexico/';
-- Confirms Gulf of Mexico SST file uploaded correctly
-- Should show SST-gulfofmexico.csv

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/';
-- Shows all folders in the volume
-- Find the exact name of the Gulf of Mexico SST folder

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/SST-GulfOfMexico/';
-- Confirms SST Gulf of Mexico file uploaded correctly
-- Should show SST-gulfofmexico.csv

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/SST-GulfOfMexico/SST-gulfofmexico.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)
LIMIT 5;
-- Shows column names and sample data
-- Confirms structure before Bronze ingestion

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_gold.dim_environment;

CREATE TABLE mcphersonsharyn_gold.dim_environment
USING DELTA AS
SELECT
  h3_res4,
  year,
  month,
  sea_surface_temperature,
  chlorophyll_concentration,
  current_speed,
  sea_level_anomaly,
  mixed_layer_depth,
  salinity,
  ocean_temperature,
  microplastic_concentration,
  microplastics_source,
  degree_heating_week,
  bleaching_alert_area,
  bleaching_alert_category,
  habitat_stress_index
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Rebuilds dim_environment with Gulf of Mexico SST coverage
-- Expected: 2,985,166 rows

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_gold.dim_environment;

CREATE TABLE mcphersonsharyn_gold.dim_environment
(
  h3_res4                    STRING,
  year                       INT,
  month                      INT,
  sea_surface_temperature    DOUBLE,
  chlorophyll_concentration  DOUBLE,
  current_speed              DOUBLE,
  sea_level_anomaly          DOUBLE,
  mixed_layer_depth          DOUBLE,
  salinity                   DOUBLE,
  ocean_temperature          DOUBLE,
  microplastic_concentration DOUBLE,
  microplastics_source       STRING,
  degree_heating_week        DOUBLE,
  bleaching_alert_area       INT,
  bleaching_alert_category   STRING,
  habitat_stress_index       INT
)
USING DELTA;
-- Empty dim_environment table with correct schema

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.dim_environment
SELECT
  h3_res4,
  year,
  month,
  sea_surface_temperature,
  chlorophyll_concentration,
  current_speed,
  sea_level_anomaly,
  mixed_layer_depth,
  salinity,
  ocean_temperature,
  microplastic_concentration,
  microplastics_source,
  degree_heating_week,
  bleaching_alert_area,
  bleaching_alert_category,
  habitat_stress_index
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Populates dim_environment from Silver
-- Expected: 2,985,166 rows

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_gold.fact_species_observations;

CREATE TABLE mcphersonsharyn_gold.fact_species_observations
(
  observation_id           BIGINT,
  species_id               INT,
  animal_id                STRING,
  date_id                  STRING,
  year                     INT,
  month                    INT,
  observation_timestamp    TIMESTAMP,
  latitude                 DOUBLE,
  longitude                DOUBLE,
  h3_res4                  STRING,
  h3_res5                  STRING,
  locality                 STRING,
  algorithm_marked_outlier BOOLEAN,
  sensor_type              STRING,
  argos_lc                 STRING
)
USING DELTA;
-- Recreating fact table to pick up updated dim_environment
-- All four species inserted separately

In [0]:
%sql
-- Reef Manta Ray
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  CAST(`event-id` AS BIGINT),
  1,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)),
  MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE),
  CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  locality,
  CAST(`manually-marked-outlier` AS BOOLEAN),
  `sensor-type`,
  NULL
FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
-- Whale Shark
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  CAST(`event-id` AS BIGINT),
  2,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)),
  MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE),
  CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  NULL,
  CAST(`algorithm-marked-outlier` AS BOOLEAN),
  `sensor-type`,
  `argos:lc`
FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
-- Sperm Whale
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  CAST(`event-id` AS BIGINT),
  3,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)),
  MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE),
  CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  NULL,
  CAST(`manually-marked-outlier` AS BOOLEAN),
  `sensor-type`,
  NULL
FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
-- Green Turtle
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  CAST(`event-id` AS BIGINT),
  4,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)),
  MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE),
  CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  NULL,
  CAST(`algorithm-marked-outlier` AS BOOLEAN),
  `sensor-type`,
  `argos:lc`
FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
SELECT
  s.species_common,
  d.enso_phase,
  count(*) AS total_observations,
  count(DISTINCT f.animal_id) AS unique_animals,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.chlorophyll_concentration), 4) AS avg_cc,
  round(avg(e.habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE d.enso_phase IS NOT NULL
GROUP BY s.species_common, d.enso_phase
ORDER BY s.species_common, avg_sst DESC;

-- All four species with ENSO phase comparison
-- Gulf of Mexico species should now show results
-- La Nina should still be warmer than El Nino for manta rays

In [0]:
%sql
SELECT
  f.longitude,
  f.latitude,
  f.h3_res4,
  f.year,
  f.month,
  e.sea_surface_temperature
FROM mcphersonsharyn_gold.fact_species_observations f
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE f.species_id = 3
LIMIT 10;
-- Checks sperm whale observations and whether they join to environment
-- Shows exact coordinates and whether H3 cells match

In [0]:
%sql
SELECT count(*) AS matching_rows
FROM mcphersonsharyn_gold.dim_environment
WHERE h3_res4 = '595679279254077439'
AND year = 2011
AND month = 7;
-- Checks if sperm whale H3 cell exists in dim_environment
-- If zero the H3 cell from the tracking data does not match
-- the H3 cells generated from SST coordinates

In [0]:
%sql
SELECT DISTINCT l.h3_res4, l.latitude, l.longitude
FROM mcphersonsharyn_gold.dim_location l
WHERE l.latitude BETWEEN 28 AND 29
AND l.longitude BETWEEN -89 AND -87
LIMIT 10;
-- Checks if SST H3 cells exist near sperm whale coordinates
-- dim_location has latitude and longitude from SST Silver
-- If empty SST has no data in this area

In [0]:
%sql
SELECT
  min(latitude) AS min_lat,
  max(latitude) AS max_lat,
  min(longitude) AS min_lon,
  max(longitude) AS max_lon
FROM mcphersonsharyn_gold.dim_location
WHERE longitude BETWEEN -97 AND -72;
-- Shows the actual coordinate range of Gulf of Mexico SST in dim_location
-- Sperm whale is at latitude 28 -- is that within our SST coverage?

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_gold.dim_environment;

CREATE TABLE mcphersonsharyn_gold.dim_environment
(
  h3_res4                    STRING,
  year                       INT,
  month                      INT,
  sea_surface_temperature    DOUBLE,
  chlorophyll_concentration  DOUBLE,
  current_speed              DOUBLE,
  sea_level_anomaly          DOUBLE,
  mixed_layer_depth          DOUBLE,
  salinity                   DOUBLE,
  ocean_temperature          DOUBLE,
  microplastic_concentration DOUBLE,
  microplastics_source       STRING,
  degree_heating_week        DOUBLE,
  bleaching_alert_area       INT,
  bleaching_alert_category   STRING,
  habitat_stress_index       INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.dim_environment
SELECT
  h3_res4, year, month,
  sea_surface_temperature, chlorophyll_concentration,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  microplastic_concentration, microplastics_source,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Rebuilds dim_environment with extended latitude coverage
-- Expected: 3,472,418 rows

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_gold.fact_species_observations;

CREATE TABLE mcphersonsharyn_gold.fact_species_observations
(
  observation_id           BIGINT,
  species_id               INT,
  animal_id                STRING,
  date_id                  STRING,
  year                     INT,
  month                    INT,
  observation_timestamp    TIMESTAMP,
  latitude                 DOUBLE,
  longitude                DOUBLE,
  h3_res4                  STRING,
  h3_res5                  STRING,
  locality                 STRING,
  algorithm_marked_outlier BOOLEAN,
  sensor_type              STRING,
  argos_lc                 STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT CAST(`event-id` AS BIGINT), 1,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)), MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE), CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  locality, CAST(`manually-marked-outlier` AS BOOLEAN), `sensor-type`, NULL
FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT CAST(`event-id` AS BIGINT), 2,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)), MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE), CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  NULL, CAST(`algorithm-marked-outlier` AS BOOLEAN), `sensor-type`, `argos:lc`
FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT CAST(`event-id` AS BIGINT), 3,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)), MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE), CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  NULL, CAST(`manually-marked-outlier` AS BOOLEAN), `sensor-type`, NULL
FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT CAST(`event-id` AS BIGINT), 4,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)), MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE), CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  NULL, CAST(`algorithm-marked-outlier` AS BOOLEAN), `sensor-type`, `argos:lc`
FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
SELECT
  s.species_common,
  d.enso_phase,
  count(*) AS total_observations,
  count(DISTINCT f.animal_id) AS unique_animals,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.chlorophyll_concentration), 4) AS avg_cc,
  round(avg(e.habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE d.enso_phase IS NOT NULL
GROUP BY s.species_common, d.enso_phase
ORDER BY s.species_common, avg_sst DESC;

-- All four species should now show results
-- La Nina should still be warmer than El Nino for manta rays

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_gold.dim_environment;

CREATE TABLE mcphersonsharyn_gold.dim_environment
(
  h3_res4                    STRING,
  year                       INT,
  month                      INT,
  sea_surface_temperature    DOUBLE,
  chlorophyll_concentration  DOUBLE,
  current_speed              DOUBLE,
  sea_level_anomaly          DOUBLE,
  mixed_layer_depth          DOUBLE,
  salinity                   DOUBLE,
  ocean_temperature          DOUBLE,
  microplastic_concentration DOUBLE,
  microplastics_source       STRING,
  degree_heating_week        DOUBLE,
  bleaching_alert_area       INT,
  bleaching_alert_category   STRING,
  habitat_stress_index       INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.dim_environment
SELECT
  h3_res4, year, month,
  sea_surface_temperature, chlorophyll_concentration,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  microplastic_concentration, microplastics_source,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Rebuilds dim_environment with full global coverage
-- Expected: 17,882,190 rows

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_gold.fact_species_observations;

CREATE TABLE mcphersonsharyn_gold.fact_species_observations
(
  observation_id           BIGINT,
  species_id               INT,
  animal_id                STRING,
  date_id                  STRING,
  year                     INT,
  month                    INT,
  observation_timestamp    TIMESTAMP,
  latitude                 DOUBLE,
  longitude                DOUBLE,
  h3_res4                  STRING,
  h3_res5                  STRING,
  locality                 STRING,
  algorithm_marked_outlier BOOLEAN,
  sensor_type              STRING,
  argos_lc                 STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT CAST(`event-id` AS BIGINT), 1,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)), MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE), CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  locality, CAST(`manually-marked-outlier` AS BOOLEAN), `sensor-type`, NULL
FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT CAST(`event-id` AS BIGINT), 2,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)), MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE), CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  NULL, CAST(`algorithm-marked-outlier` AS BOOLEAN), `sensor-type`, `argos:lc`
FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT CAST(`event-id` AS BIGINT), 3,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)), MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE), CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  NULL, CAST(`manually-marked-outlier` AS BOOLEAN), `sensor-type`, NULL
FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT CAST(`event-id` AS BIGINT), 4,
  CAST(`individual-local-identifier` AS STRING),
  CONCAT(CAST(YEAR(TO_TIMESTAMP(`timestamp`)) AS STRING), '-',
    LPAD(CAST(MONTH(TO_TIMESTAMP(`timestamp`)) AS STRING), 2, '0')),
  YEAR(TO_TIMESTAMP(`timestamp`)), MONTH(TO_TIMESTAMP(`timestamp`)),
  TO_TIMESTAMP(`timestamp`),
  CAST(`location-lat` AS DOUBLE), CAST(`location-long` AS DOUBLE),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4),
  h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5),
  NULL, CAST(`algorithm-marked-outlier` AS BOOLEAN), `sensor-type`, `argos:lc`
FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
WHERE `location-lat` IS NOT NULL AND `location-long` IS NOT NULL;

In [0]:
%sql
SELECT
  s.species_common,
  d.enso_phase,
  count(*) AS total_observations,
  count(DISTINCT f.animal_id) AS unique_animals,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.chlorophyll_concentration), 4) AS avg_cc,
  round(avg(e.habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE d.enso_phase IS NOT NULL
GROUP BY s.species_common, d.enso_phase
ORDER BY s.species_common, avg_sst DESC;
-- THE KEY ATLAS FINDING ON COMPLETE GLOBAL COVERAGE
-- All four species with full SST coverage
-- Green turtle should now show results
-- La Nina should still be warmer than El Nino for manta rays

In [0]:
%sql

-- SECTION 3 — DIM_LOCATION REBUILD
-- Rebuilding with full global coverage including Indian Ocean
-- One row per unique H3 resolution 4 cell in SST Silver
-- Enriched with geographic context for all four species habitats


DROP TABLE IF EXISTS mcphersonsharyn_gold.dim_location;

CREATE TABLE mcphersonsharyn_gold.dim_location
USING DELTA AS
SELECT DISTINCT
  sst.h3_res4,
  sst.h3_res5,
  h3_longlatash3(sst.longitude, sst.latitude, 3) AS h3_res3,
  sst.latitude,
  sst.longitude,
  CASE
    WHEN sst.longitude BETWEEN -98 AND -80
      AND sst.latitude BETWEEN 18 AND 31
      THEN 'Gulf of Mexico'
    WHEN sst.longitude BETWEEN 125 AND 135
      AND sst.latitude BETWEEN -5 AND 2
      THEN 'Raja Ampat Indonesia'
    WHEN sst.longitude BETWEEN 68 AND 75
      AND sst.latitude BETWEEN -8 AND -3
      THEN 'Chagos Archipelago'
    WHEN sst.longitude BETWEEN 30 AND 80
      AND sst.latitude BETWEEN -25 AND 10
      THEN 'Western Indian Ocean'
    WHEN sst.longitude BETWEEN 80 AND 125
      AND sst.latitude BETWEEN -10 AND 22
      THEN 'Eastern Indian Ocean'
    WHEN sst.longitude BETWEEN -180 AND -73
      THEN 'Western Atlantic and Caribbean'
    WHEN sst.longitude BETWEEN 126 AND 180
      THEN 'Indo-Pacific'
    ELSE 'Open Ocean'
  END AS study_region,
  CASE
    WHEN sst.longitude BETWEEN -180 AND -73
      THEN 'Atlantic'
    WHEN sst.longitude BETWEEN -72 AND 30
      THEN 'Atlantic'
    WHEN sst.longitude BETWEEN 30 AND 125
      THEN 'Indian Ocean'
    WHEN sst.longitude BETWEEN 125 AND 180
      THEN 'Indo-Pacific'
    ELSE 'Pacific'
  END AS ocean_basin,
  CASE
    WHEN sst.longitude BETWEEN 125 AND 135
      AND sst.latitude BETWEEN -5 AND 2
      THEN 'Reef Manta Ray'
    WHEN sst.longitude BETWEEN -98 AND -80
      AND sst.latitude BETWEEN 18 AND 31
      THEN 'Whale Shark and Sperm Whale'
    WHEN sst.longitude BETWEEN 30 AND 80
      AND sst.latitude BETWEEN -25 AND 10
      THEN 'Green Turtle'
    ELSE NULL
  END AS primary_species
FROM mcphersonsharyn_silver.sst_silver sst
ORDER BY sst.latitude, sst.longitude;
-- Rebuilt from SST Silver which now includes Indian Ocean
-- Chagos Archipelago now has H3 cells
-- primary_species identifies core habitat cells for each species
-- study_region gives geographic context for dashboard labels

In [0]:
%sql
SELECT
  ocean_basin,
  study_region,
  count(*) AS h3_cells
FROM mcphersonsharyn_gold.dim_location
GROUP BY ocean_basin, study_region
ORDER BY ocean_basin, study_region;
-- Confirms Indian Ocean and Chagos cells now present
-- Should show all four species habitat regions

In [0]:
%sql

-- ANALYSIS 1 — ENSO PHASE SST COMPARISON — ALL FOUR SPECIES


SELECT
  s.species_common,
  s.ocean_basin,
  d.enso_phase,
  count(*) AS total_observations,
  count(DISTINCT f.animal_id) AS unique_animals,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(min(e.sea_surface_temperature), 2) AS min_sst,
  round(max(e.sea_surface_temperature), 2) AS max_sst,
  round(avg(e.chlorophyll_concentration), 4) AS avg_cc,
  round(avg(e.degree_heating_week), 2) AS avg_dhw,
  round(avg(e.habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE d.enso_phase IS NOT NULL
GROUP BY s.species_common, s.ocean_basin, d.enso_phase
ORDER BY s.species_common, avg_sst DESC;
-- Complete ENSO analysis across all four species
-- Shows whether La Nina is warmer than El Nino for each species
-- Raja Ampat inversion should be clear vs expected pattern elsewhere

In [0]:
%sql

-- ANALYSIS 2 — LONG TERM SST TREND BY SPECIES
-- Shows year on year warming in each species habitat

SELECT
  s.species_common,
  f.year,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.habitat_stress_index), 2) AS avg_stress,
  d.enso_phase
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE e.sea_surface_temperature IS NOT NULL
GROUP BY s.species_common, f.year, d.enso_phase
ORDER BY s.species_common, f.year;
-- Year on year SST in each species habitat
-- Shows long term warming trend
-- ENSO phase shown alongside to separate climate signal from noise

In [0]:
%sql

-- ANALYSIS 3 — HABITAT STRESS INDEX BY SPECIES AND REGION
-- Compares cumulative environmental stress across ocean basins

SELECT
  s.species_common,
  l.ocean_basin,
  l.study_region,
  count(*) AS total_observations,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.habitat_stress_index), 2) AS avg_stress,
  count(CASE WHEN e.habitat_stress_index >= 2 THEN 1 END) AS high_stress_obs,
  round(count(CASE WHEN e.habitat_stress_index >= 2 THEN 1 END) * 100.0 / count(*), 1) AS high_stress_pct,
  round(avg(e.degree_heating_week), 2) AS avg_dhw,
  round(avg(e.current_speed), 4) AS avg_current_speed
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
JOIN mcphersonsharyn_gold.dim_location l
  ON f.h3_res4 = l.h3_res4
WHERE e.sea_surface_temperature IS NOT NULL
GROUP BY s.species_common, l.ocean_basin, l.study_region
ORDER BY avg_stress DESC;
-- Compares habitat stress across all four species and regions
-- high_stress_pct shows what proportion of time animals experience HSI >= 2
-- Directly answers: are Gulf of Mexico species under more stress than Indian Ocean?

In [0]:
%sql

-- ANALYSIS 4 — ANIMALS IN HIGH STRESS HABITAT
-- Which animals were present when habitat stress index was highest?
-- These are the highest priority conservation findings in ATLAS

SELECT
  s.species_common,
  f.animal_id,
  f.year,
  f.month,
  d.month_name,
  d.enso_phase,
  l.study_region,
  round(e.sea_surface_temperature, 2) AS sst,
  round(e.chlorophyll_concentration, 4) AS cc,
  round(e.degree_heating_week, 2) AS dhw,
  e.bleaching_alert_category,
  e.habitat_stress_index
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
JOIN mcphersonsharyn_gold.dim_location l ON f.h3_res4 = l.h3_res4
WHERE e.habitat_stress_index >= 2
AND e.sea_surface_temperature IS NOT NULL
ORDER BY e.habitat_stress_index DESC, e.sea_surface_temperature DESC
LIMIT 20;
-- Shows individual animals present during highest stress conditions
-- HSI >= 2 means at least two simultaneous stressors
-- These specific animals in specific months are the conservation story
-- animal_id identifies the individual animal by its tracking tag

In [0]:
%sql
SELECT DISTINCT
  s.species_common,
  f.animal_id,
  f.year,
  f.month,
  d.month_name,
  d.enso_phase,
  l.study_region,
  round(e.sea_surface_temperature, 2) AS sst,
  round(e.chlorophyll_concentration, 4) AS cc,
  round(e.degree_heating_week, 2) AS dhw,
  e.bleaching_alert_category,
  e.habitat_stress_index
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
JOIN mcphersonsharyn_gold.dim_location l ON f.h3_res4 = l.h3_res4
WHERE e.habitat_stress_index >= 2
AND e.sea_surface_temperature IS NOT NULL
ORDER BY habitat_stress_index DESC, sst DESC
LIMIT 20;
-- Fixed: ORDER BY uses alias names not table.column references
-- DISTINCT removes duplicate rows from multiple fact observations
-- per H3 cell per month

In [0]:
%sql
-- Check 1: Are there duplicate H3 + year + month in dim_environment?
SELECT
  h3_res4, year, month,
  count(*) AS row_count
FROM mcphersonsharyn_gold.dim_environment
GROUP BY h3_res4, year, month
HAVING count(*) > 1
ORDER BY row_count DESC
LIMIT 5;
-- If this returns rows dim_environment has duplicates
-- If empty the duplicates are in fact_species_observations

In [0]:
%sql
-- Check 2: Are there duplicate observations in fact table?
SELECT
  species_id, animal_id, year, month, h3_res4,
  count(*) AS row_count
FROM mcphersonsharyn_gold.fact_species_observations
GROUP BY species_id, animal_id, year, month, h3_res4
HAVING count(*) > 1
ORDER BY row_count DESC
LIMIT 5;
-- If this returns rows the fact table has duplicate observations
-- Multiple GPS fixes in same H3 cell same month for same animal

In [0]:
%sql

-- NOTE — FACT TABLE DESIGN DECISION
-- fact_species_observations contains one row per GPS tracking fix
-- A single animal may have hundreds of fixes in the same H3 cell in the same month — this is correct and scientifically valuable
--
-- Example: Green Turtle 61813 — 535 fixes in one H3 cell in Dec 2012.This tells us the animal spent significant time in that location
--
-- ALWAYS use DISTINCT or GROUP BY in analysis queries to avoid counting the same environmental conditions multiple times
-- This is standard star schema analytical practice
--
-- DO NOT aggregate the fact table to monthly — daily GPS resolution is scientifically valuable for movement analysis


In [0]:
%sql
-- Rebuild fact_species_observations from Silver not Bronze
-- Correct medallion architecture: Bronze → Silver → Gold
DROP TABLE IF EXISTS mcphersonsharyn_gold.fact_species_observations;

CREATE TABLE mcphersonsharyn_gold.fact_species_observations
(
  observation_id           BIGINT,
  species_id               INT,
  animal_id                STRING,
  date_id                  STRING,
  year                     INT,
  month                    INT,
  observation_timestamp    TIMESTAMP,
  latitude                 DOUBLE,
  longitude                DOUBLE,
  h3_res4                  STRING,
  h3_res5                  STRING,
  locality                 STRING,
  algorithm_marked_outlier BOOLEAN,
  sensor_type              STRING,
  argos_lc                 STRING
)
USING DELTA;
-- Rebuilt to read from Silver species tables not Bronze
-- Correct medallion architecture Bronze → Silver → Gold
-- Silver already has H3 indexing, outlier filtering and clean columns

In [0]:
%sql
-- Reef Manta Ray from Silver
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  event_id, species_id, animal_id, date_id,
  year, month, observation_timestamp,
  latitude, longitude, h3_res4, h3_res5,
  locality, is_outlier, sensor_type, NULL
FROM mcphersonsharyn_silver.manta_ray_silver;
-- Reading from Silver not Bronze
-- H3 already calculated in Silver
-- Expected: 3,224 rows

In [0]:
%sql
-- Whale Shark from Silver
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  event_id, species_id, animal_id, date_id,
  year, month, observation_timestamp,
  latitude, longitude, h3_res4, h3_res5,
  NULL, is_outlier, sensor_type, argos_location_class
FROM mcphersonsharyn_silver.whale_shark_silver;
-- Expected: 3,382 rows

In [0]:
%sql
-- Sperm Whale from Silver
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  event_id, species_id, animal_id, date_id,
  year, month, observation_timestamp,
  latitude, longitude, h3_res4, h3_res5,
  NULL, is_outlier, sensor_type, NULL
FROM mcphersonsharyn_silver.sperm_whale_silver;
-- Expected: 7,759 rows

In [0]:
%sql
-- Green Turtle from Silver
INSERT INTO mcphersonsharyn_gold.fact_species_observations
SELECT
  event_id, species_id, animal_id, date_id,
  year, month, observation_timestamp,
  latitude, longitude, h3_res4, h3_res5,
  NULL, is_outlier, sensor_type, argos_location_class
FROM mcphersonsharyn_silver.green_turtle_silver;
-- Expected: 55,411 rows

In [0]:
%sql
SELECT
  s.species_common,
  count(*) AS total_observations,
  count(DISTINCT f.animal_id) AS unique_animals,
  min(f.year) AS earliest_year,
  max(f.year) AS latest_year
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
GROUP BY s.species_common
ORDER BY s.species_common;
-- Confirms all four species inserted correctly from Silver
-- Row counts should match Silver species tables exactly

In [0]:
%sql
SELECT
  s.species_common,
  d.enso_phase,
  count(DISTINCT f.animal_id) AS unique_animals,
  count(*) AS total_observations,
  round(avg(e.sea_surface_temperature), 2) AS avg_sst,
  round(avg(e.chlorophyll_concentration), 4) AS avg_cc,
  round(avg(e.habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date d ON f.date_id = d.date_id
JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4
  AND f.year = e.year
  AND f.month = e.month
WHERE d.enso_phase IS NOT NULL
GROUP BY s.species_common, d.enso_phase
ORDER BY s.species_common, avg_sst DESC;
-- Final confirmation of key finding on complete rebuilt pipeline
-- Bronze → Silver → Gold architecture confirmed

In [0]:
%sql
-- Show all tables in your Gold schema
SHOW TABLES IN mcphersonsharyn_gold;

In [0]:
%sql
DESCRIBE mcphersonsharyn_gold.fact_species_observations;


In [0]:
%sql
DESCRIBE mcphersonsharyn_gold.dim_environment;


In [0]:
%sql
DESCRIBE mcphersonsharyn_gold.dim_location;


In [0]:
%sql
DESCRIBE mcphersonsharyn_gold.dim_species;


In [0]:
%sql
DESCRIBE mcphersonsharyn_gold.dim_date;

In [0]:
%sql
-- ═══════════════════════════════════════════════════════════════════════════
--
--   ATLAS — Animal Tracking and Lifecycle Analysis System
--   STAR SCHEMA — Full Analysis Query
--   Species observations enriched with environment, location, date and ENSO
--
-- ═══════════════════════════════════════════════════════════════════════════

SELECT

  -- Species dimension
  s.species_common,
  s.species_latin,
  s.iucn_status,
  s.feeding_type,
  s.ocean_basin,

  -- Date dimension
  d.year,
  d.month_name,
  d.quarter,
  d.season,
  d.enso_phase,
  d.mei_value,
  d.is_el_nino,
  d.is_la_nina,

  -- Location dimension
  l.study_region,
  l.ocean_basin                          AS observation_basin,

  -- Fact table
  f.animal_id,
  f.locality,
  f.latitude,
  f.longitude,
  f.sensor_type,

  -- Environment dimension
  e.sea_surface_temperature              AS sst,
  e.chlorophyll_concentration            AS chlorophyll,
  e.current_speed,
  e.sea_level_anomaly,
  e.mixed_layer_depth,
  e.salinity,
  e.microplastic_concentration,
  e.microplastics_source,
  e.degree_heating_week,
  e.bleaching_alert_category,
  e.habitat_stress_index

FROM mcphersonsharyn_gold.fact_species_observations  f

-- Species
JOIN mcphersonsharyn_gold.dim_species     s
  ON f.species_id = s.species_id

-- Date and ENSO
JOIN mcphersonsharyn_gold.dim_date        d
  ON f.date_id = d.date_id

-- Environment (H3 + year + month)
LEFT JOIN mcphersonsharyn_gold.dim_environment  e
  ON  f.h3_res4 = e.h3_res4
  AND f.year    = e.year
  AND f.month   = e.month

-- Location (H3 cell attributes)
LEFT JOIN mcphersonsharyn_gold.dim_location  l
  ON f.h3_res4 = l.h3_res4

WHERE e.sea_surface_temperature IS NOT NULL

ORDER BY
  s.species_common,
  d.year,
  d.month_name,
  f.animal_id;

In [0]:
%sql

-- Quick summary: rows, species, ENSO phases, null rates
SELECT
  s.species_common,
  d.enso_phase,
  d.year,
  count(*)                                                    AS observations,
  count(*) filter(WHERE e.sea_surface_temperature IS NULL)    AS null_sst,
  count(*) filter(WHERE e.chlorophyll_concentration IS NULL)  AS null_chlorophyll,
  round(avg(e.sea_surface_temperature), 2)                    AS avg_sst,
  round(avg(e.chlorophyll_concentration), 3)                  AS avg_chlorophyll,
  round(avg(e.habitat_stress_index), 2)                       AS avg_stress

FROM mcphersonsharyn_gold.fact_species_observations  f
JOIN mcphersonsharyn_gold.dim_species       s  ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date          d  ON f.date_id = d.date_id
LEFT JOIN mcphersonsharyn_gold.dim_environment  e
  ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month

GROUP BY s.species_common, d.enso_phase, d.year
ORDER BY s.species_common, d.year, d.enso_phase;

In [0]:
%sql
-- Which H3 cells in dim_environment have CC data vs null?
-- Shows where the CC coverage gap is geographically
SELECT
  CASE
    WHEN chlorophyll_concentration IS NOT NULL THEN 'Has CC data'
    ELSE 'No CC data'
  END                              AS cc_coverage,
  count(DISTINCT h3_res4)          AS distinct_h3_cells,
  round(min(sea_surface_temperature), 1) AS min_sst,
  round(max(sea_surface_temperature), 1) AS max_sst,
  count(*)                         AS total_rows
FROM mcphersonsharyn_gold.dim_environment
GROUP BY cc_coverage;

In [0]:
%sql
SELECT
  round(min(l.latitude), 1)   AS min_lat,
  round(max(l.latitude), 1)   AS max_lat,
  round(min(l.longitude), 1)  AS min_lon,
  round(max(l.longitude), 1)  AS max_lon,
  count(DISTINCT e.h3_res4)   AS cells_with_cc
FROM mcphersonsharyn_gold.dim_environment  e
JOIN mcphersonsharyn_gold.dim_location     l  ON e.h3_res4 = l.h3_res4
WHERE e.chlorophyll_concentration IS NOT NULL;

In [0]:
%sql
-- How many distinct H3 cells are in cc_silver?
SELECT
  count(DISTINCT h3_res4)  AS distinct_h3_cells,
  count(*)                 AS total_rows,
  min(year)                AS min_year,
  max(year)                AS max_year
FROM mcphersonsharyn_silver.cc_silver;

In [0]:
%sql
--How many distinct H3 cells are in the fact table (species observations)?
SELECT
  count(DISTINCT h3_res4)  AS distinct_h3_cells
FROM mcphersonsharyn_gold.fact_species_observations;


In [0]:
%sql
SELECT
  count(DISTINCT h3_res4)  AS distinct_h3_cells
FROM mcphersonsharyn_gold.fact_species_observations;

In [0]:
%sql
SELECT
  count(DISTINCT f.h3_res4)  AS species_cells_with_cc_match
FROM mcphersonsharyn_gold.fact_species_observations  f
WHERE f.h3_res4 IN (
  SELECT DISTINCT h3_res4
  FROM mcphersonsharyn_silver.cc_silver
);

In [0]:
%sql
-- Sample of H3 cells from species observations
SELECT DISTINCT h3_res4 FROM mcphersonsharyn_gold.fact_species_observations LIMIT 5;

In [0]:
%sql
SELECT DISTINCT h3_res4 FROM mcphersonsharyn_silver.cc_silver LIMIT 5;


In [0]:
%sql
-- Where are the species observation cells geographically?
SELECT
  s.species_common,
  round(avg(f.latitude), 1)   AS avg_lat,
  round(avg(f.longitude), 1)  AS avg_lon,
  round(min(f.latitude), 1)   AS min_lat,
  round(max(f.latitude), 1)   AS max_lat,
  round(min(f.longitude), 1)  AS min_lon,
  round(max(f.longitude), 1)  AS max_lon,
  count(DISTINCT f.h3_res4)   AS h3_cells
FROM mcphersonsharyn_gold.fact_species_observations  f
JOIN mcphersonsharyn_gold.dim_species  s  ON f.species_id = s.species_id
GROUP BY s.species_common;

In [0]:
%sql
-- Where is the CC data geographically?
SELECT
  round(avg(latitude), 1)   AS avg_lat,
  round(avg(longitude), 1)  AS avg_lon,
  round(min(latitude), 1)   AS min_lat,
  round(max(latitude), 1)   AS max_lat,
  round(min(longitude), 1)  AS min_lon,
  round(max(longitude), 1)  AS max_lon
FROM mcphersonsharyn_silver.cc_silver;

In [0]:
%sql
SELECT
  count(*)              AS rows,
  count(DISTINCT h3_res4) AS h3_cells
FROM mcphersonsharyn_silver.cc_silver
WHERE longitude BETWEEN -98 AND -83
AND latitude BETWEEN 19 AND 31;


In [0]:
%sql
-- CC cells in Gulf of Mexico
SELECT DISTINCT h3_res4, round(latitude,2) AS lat, round(longitude,2) AS lon
FROM mcphersonsharyn_silver.cc_silver
WHERE longitude BETWEEN -98 AND -83
AND latitude BETWEEN 19 AND 31
LIMIT 10;

In [0]:
%sql
-- Whale shark and sperm whale cells in same region
SELECT DISTINCT f.h3_res4, round(f.latitude,2) AS lat, round(f.longitude,2) AS lon
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
WHERE s.species_common IN ('Whale Shark', 'Sperm Whale')
LIMIT 10;

In [0]:
%sql
-- What years/months does CC silver have for Gulf of Mexico cells?
SELECT year, month, count(*) AS rows
FROM mcphersonsharyn_silver.cc_silver
WHERE longitude BETWEEN -98 AND -83
AND latitude BETWEEN 19 AND 31
GROUP BY year, month
ORDER BY year, month;

In [0]:
%sql
-- What years/months do whale shark and sperm whale observations have?
SELECT d.year, d.month, count(*) AS observations
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_date   d ON f.date_id = d.date_id
WHERE s.species_common IN ('Whale Shark', 'Sperm Whale')
GROUP BY d.year, d.month
ORDER BY d.year, d.month;

In [0]:
%sql
-- Direct join test: how many whale shark/sperm whale observations
-- successfully match a CC row on h3 + year + month?
SELECT
  s.species_common,
  f.year,
  f.month,
  count(*)                                              AS observations,
  count(c.h3_res4)                                     AS matched_to_cc,
  count(*) - count(c.h3_res4)                          AS unmatched
FROM mcphersonsharyn_gold.fact_species_observations  f
JOIN mcphersonsharyn_gold.dim_species  s  ON f.species_id = s.species_id
LEFT JOIN mcphersonsharyn_silver.cc_silver  c
  ON  f.h3_res4 = c.h3_res4
  AND f.year    = c.year
  AND f.month   = c.month
WHERE s.species_common IN ('Whale Shark', 'Sperm Whale')
GROUP BY s.species_common, f.year, f.month
ORDER BY s.species_common, f.year, f.month;

In [0]:
%sql
-- A whale shark observation: what H3 cell does it produce?
SELECT
  animal_id,
  latitude,
  longitude,
  h3_res4,
  h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 4) AS h3_recalc_correct,
  h3_longlatash3(CAST(latitude AS DOUBLE), CAST(longitude AS DOUBLE), 4) AS h3_recalc_swapped
FROM mcphersonsharyn_gold.fact_species_observations
LIMIT 3;

In [0]:
%sql
-- A CC silver row in Gulf of Mexico: what H3 cell does it produce?
SELECT
  latitude,
  longitude,
  h3_res4,
  h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 4) AS h3_recalc_correct,
  h3_longlatash3(CAST(latitude AS DOUBLE), CAST(longitude AS DOUBLE), 4) AS h3_recalc_swapped
FROM mcphersonsharyn_silver.cc_silver
WHERE longitude BETWEEN -98 AND -83
AND latitude BETWEEN 19 AND 31
LIMIT 3;

In [0]:
%sql
-- Take 5 whale shark H3 cells and see what CC cells are nearby
SELECT DISTINCT
  f.h3_res4                                    AS species_h3,
  f.latitude                                   AS species_lat,
  f.longitude                                  AS species_lon,
  c.h3_res4                                    AS cc_h3,
  c.latitude                                   AS cc_lat,
  c.longitude                                  AS cc_lon,
  abs(f.latitude  - c.latitude)                AS lat_diff,
  abs(f.longitude - c.longitude)               AS lon_diff
FROM mcphersonsharyn_gold.fact_species_observations  f
JOIN mcphersonsharyn_gold.dim_species  s  ON f.species_id = s.species_id
JOIN mcphersonsharyn_silver.cc_silver  c
  ON  abs(f.latitude  - c.latitude)  < 2
  AND abs(f.longitude - c.longitude) < 2
  AND f.year  = c.year
  AND f.month = c.month
WHERE s.species_common = 'Whale Shark'
LIMIT 10;

In [0]:
%sql
-- Test: does res3 produce matching cells for a whale shark and nearby CC point?
SELECT
  'Whale Shark'                                                           AS source,
  23.64545935                                                             AS lat,
  -96.66867731                                                            AS lon,
  h3_longlatash3(-96.66867731, 23.64545935, 3)                           AS h3_res3

UNION ALL

SELECT
  'CC grid centre',
  22,
  -96,
  h3_longlatash3(-96, 22, 3);

In [0]:
%sql
SELECT
  'Whale Shark'                                AS source,
  23.64545935                                  AS lat,
  -96.66867731                                 AS lon,
  h3_longlatash3(-96.66867731, 23.64545935, 2) AS h3_res2

UNION ALL

SELECT
  'CC grid centre',
  22,
  -96,
  h3_longlatash3(-96, 22, 2);

In [0]:
%sql
-- How many whale shark observations match CC at res2 + year + month?
SELECT
  count(*)              AS observations,
  count(c.h3_res2)      AS matched_to_cc,
  count(*) - count(c.h3_res2) AS unmatched
FROM mcphersonsharyn_gold.fact_species_observations  f
JOIN mcphersonsharyn_gold.dim_species  s  ON f.species_id = s.species_id
LEFT JOIN (
  SELECT
    h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 2) AS h3_res2,
    year,
    month,
    avg(chlorophyll_concentration) AS chlorophyll_concentration
  FROM mcphersonsharyn_silver.cc_silver
  GROUP BY 1, 2, 3
) c
  ON  h3_longlatash3(CAST(f.longitude AS DOUBLE), CAST(f.latitude AS DOUBLE), 2) = c.h3_res2
  AND f.year  = c.year
  AND f.month = c.month
WHERE s.species_common = 'Whale Shark';

In [0]:
%sql
-- Which year/months are getting CC matches at res2?
SELECT
  f.year,
  f.month,
  count(*)              AS observations,
  count(c.h3_res2)      AS matched_to_cc
FROM mcphersonsharyn_gold.fact_species_observations  f
JOIN mcphersonsharyn_gold.dim_species  s  ON f.species_id = s.species_id
LEFT JOIN (
  SELECT
    h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 2) AS h3_res2,
    year,
    month,
    avg(chlorophyll_concentration) AS chlorophyll_concentration
  FROM mcphersonsharyn_silver.cc_silver
  GROUP BY 1, 2, 3
) c
  ON  h3_longlatash3(CAST(f.longitude AS DOUBLE), CAST(f.latitude AS DOUBLE), 2) = c.h3_res2
  AND f.year  = c.year
  AND f.month = c.month
WHERE s.species_common = 'Whale Shark'
GROUP BY f.year, f.month
ORDER BY f.year, f.month;

In [0]:
%sql
-- Which whale shark H3 res2 cells have NO corresponding CC res2 cell?
-- This tells us: data coverage gap vs resolution mismatch
SELECT
  h3_longlatash3(CAST(f.longitude AS DOUBLE), CAST(f.latitude AS DOUBLE), 2) AS species_h3_res2,
  round(avg(f.latitude), 2)  AS avg_lat,
  round(avg(f.longitude), 2) AS avg_lon,
  count(*)                   AS observations
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
WHERE s.species_common = 'Whale Shark'
AND h3_longlatash3(CAST(f.longitude AS DOUBLE), CAST(f.latitude AS DOUBLE), 2)
    NOT IN (
      SELECT DISTINCT h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 2)
      FROM mcphersonsharyn_silver.cc_silver
    )
GROUP BY 1
ORDER BY observations DESC;

In [0]:
%sql
-- What's the actual lon/lat range of CC data in the Gulf region?
SELECT 
  round(min(longitude), 1) AS min_lon,
  round(max(longitude), 1) AS max_lon,
  round(min(latitude), 1)  AS min_lat,
  round(max(latitude), 1)  AS max_lat,
  count(*) AS rows
FROM mcphersonsharyn_silver.cc_silver
WHERE longitude BETWEEN -100 AND -80
AND latitude BETWEEN 18 AND 32;

In [0]:
%sql
SELECT year, count(*) AS rows
FROM mcphersonsharyn_silver.cc_silver
WHERE longitude BETWEEN -100 AND -80
AND latitude BETWEEN 18 AND 22
GROUP BY year
ORDER BY year;


In [0]:
%sql
SELECT
  s.species_common,
  f.latitude,
  f.longitude,
  f.year,
  f.month,
  e.sea_surface_temperature AS sst
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
WHERE e.sea_surface_temperature IS NOT NULL
ORDER BY s.species_common, f.year, f.month;

In [0]:
%sql
SELECT * FROM (
  SELECT
    s.species_common,
    ROUND(f.latitude, 2)  AS latitude,
    ROUND(f.longitude, 2) AS longitude,
    f.year,
    f.month,
    e.sea_surface_temperature AS sst,
    ROW_NUMBER() OVER (PARTITION BY s.species_common, f.year, f.month ORDER BY f.latitude) AS rn
  FROM mcphersonsharyn_gold.fact_species_observations f
  JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
  LEFT JOIN mcphersonsharyn_gold.dim_environment e
    ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
  WHERE e.sea_surface_temperature IS NOT NULL
) WHERE MOD(rn, 10) = 1
ORDER BY species_common, year, month;

In [0]:
%sql
-- ATLAS Animation Export: ENSO Phase Overlay
-- Samples ~1 in 10 observations per species/year/month
-- Joins ENSO phase and MEI value from dim_date for background colour coding
-- Export as: atlas_enso_animation.csv

SELECT * FROM (
  SELECT
    s.species_common,
    ROUND(f.latitude, 2)  AS latitude,
    ROUND(f.longitude, 2) AS longitude,
    f.year,
    f.month,
    e.sea_surface_temperature AS sst,
    d.enso_phase,       -- El Niño / La Niña / Neutral
    d.mei_value,        -- Raw MEI.v2 index value
    ROW_NUMBER() OVER (
      PARTITION BY s.species_common, f.year, f.month
      ORDER BY f.latitude
    ) AS rn
  FROM mcphersonsharyn_gold.fact_species_observations f
  JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
  JOIN mcphersonsharyn_gold.dim_date d ON f.year = d.year AND f.month = d.month
  LEFT JOIN mcphersonsharyn_gold.dim_environment e
    ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
  WHERE e.sea_surface_temperature IS NOT NULL
)
WHERE MOD(rn, 10) = 1
ORDER BY species_common, year, month;

In [0]:
%sql
-- ATLAS Animation Export: Habitat Stress Index
-- Samples ~1 in 10 observations per species/year/month
-- Colours each dot 0-4 by cumulative stress score (thermal/food/pollution/bleaching)
-- Export as: atlas_stress_animation.csv

SELECT * FROM (
  SELECT
    s.species_common,
    ROUND(f.latitude, 2)  AS latitude,
    ROUND(f.longitude, 2) AS longitude,
    f.year,
    f.month,
    e.sea_surface_temperature AS sst,
    e.habitat_stress_index,   -- 0 = no stress, 4 = all four stressors present
    ROW_NUMBER() OVER (
      PARTITION BY s.species_common, f.year, f.month
      ORDER BY f.latitude
    ) AS rn
  FROM mcphersonsharyn_gold.fact_species_observations f
  JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
  LEFT JOIN mcphersonsharyn_gold.dim_environment e
    ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
  WHERE e.habitat_stress_index IS NOT NULL
)
WHERE MOD(rn, 10) = 1
ORDER BY species_common, year, month;

In [0]:
%sql
-- ATLAS Animation Export: Green Turtle Full Resolution
-- All observations with SST -- no sampling, full 5,500+ row dataset
-- Used for single-species deep dive showing Chagos clustering and seasonal shift
-- Export as: atlas_turtle_deepdive.csv

SELECT
  ROUND(f.latitude, 2)  AS latitude,
  ROUND(f.longitude, 2) AS longitude,
  f.year,
  f.month,
  e.sea_surface_temperature AS sst
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s ON f.species_id = s.species_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
  ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
WHERE s.species_common = 'Green Turtle'
  AND e.sea_surface_temperature IS NOT NULL
ORDER BY year, month;

In [0]:
%sql
-- Option 1: see all your schemas
SHOW SCHEMAS;



In [0]:
%sql
SHOW TABLES IN mcphersonsharyn_gold;


In [0]:
%sql
-- ATLAS Green Turtle Deep Dive — full resolution, all years 2012–2019
-- Export as: ATLAS_Green_Turtle_Deepdive_Full.numbers
SELECT
    f.latitude,
    f.longitude,
    f.year,
    f.month,
    e.sea_surface_temperature AS sst
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s
    ON f.species_id = s.species_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
    ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
WHERE s.species_common = 'Green Turtle'
ORDER BY f.year, f.month, f.latitude, f.longitude


In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_gold.turtle_deepdive_export AS
SELECT
    f.latitude,
    f.longitude,
    f.year,
    f.month,
    e.sea_surface_temperature AS sst
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s
    ON f.species_id = s.species_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
    ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
WHERE s.species_common = 'Green Turtle'
ORDER BY f.year, f.month, f.latitude, f.longitude;

SELECT COUNT(*) FROM mcphersonsharyn_gold.turtle_deepdive_export;


In [0]:
%sql
-- Write full Green Turtle dataset to export table
CREATE OR REPLACE TABLE mcphersonsharyn_gold.turtle_deepdive_export AS
SELECT
    f.latitude,
    f.longitude,
    f.year,
    f.month,
    e.sea_surface_temperature AS sst
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s
    ON f.species_id = s.species_id
LEFT JOIN mcphersonsharyn_gold.dim_environment e
    ON f.h3_res4 = e.h3_res4 AND f.year = e.year AND f.month = e.month
WHERE s.species_common = 'Green Turtle'
ORDER BY f.year, f.month, f.latitude, f.longitude;

In [0]:
# Write Green Turtle deep dive to CSV in your volume
df = spark.table("mcphersonsharyn_gold.turtle_deepdive_export")

df.toPandas().to_csv(
    "/Volumes/workspace/mcphersonsharyn_gold/exports/turtle_deepdive_full.csv",
    index=False
)

In [0]:
import pandas as pd

df = spark.table("mcphersonsharyn_gold.turtle_deepdive_export").toPandas()
print(len(df))  # should be 55,411
display(df.head())


In [0]:
%sql
DESCRIBE mcphersonsharyn_gold.fact_species_observations;


In [0]:
%sql
DESCRIBE mcphersonsharyn_gold.dim_environment;

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_gold.turtle_deepdive_export AS
SELECT 
    f.latitude, 
    f.longitude, 
    f.year, 
    f.month, 
    e.sea_surface_temperature AS sst
FROM mcphersonsharyn_gold.fact_species_observations f
JOIN mcphersonsharyn_gold.dim_species s
    ON f.species_id = s.species_id
JOIN mcphersonsharyn_gold.dim_environment e
    ON f.h3_res4 = e.h3_res4
    AND f.year = e.year
    AND f.month = e.month
WHERE s.species_common = 'Green Turtle'
ORDER BY f.year, f.month;

In [0]:
%sql
SELECT COUNT(*) FROM mcphersonsharyn_gold.turtle_deepdive_export;

In [0]:
%sql
SELECT * FROM mcphersonsharyn_gold.turtle_deepdive_export LIMIT 60000